# Fine-tuning del detector de bovinos HMS (vista cenital con drone)

Este notebook entrena un modelo YOLO **propio** con las fotos y videos de tu drone,
para que cuente bovinos mejor que el modelo genérico.

**Cómo usarlo (no hace falta saber programar):**
1. Arriba a la derecha: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)** y guardar.
2. Apretá el botón de **play** de cada celda, **en orden, de arriba hacia abajo**, esperando que cada una termine.
3. Cuando la celda 2 te lo pida, subí el archivo `dataset_bovinos_limpio.zip` que generaste en tu Mac con `python scripts/preparar_dataset.py (ya generado: 61 imágenes, 1024 bovinos etiquetados a mano)`.
4. El entrenamiento (celda 3) tarda entre 30 y 90 minutos. Dejá la pestaña abierta.
5. Al final se descarga solo el archivo `hms_bovinos.pt`: **guardalo en la raíz de la carpeta del proyecto** en tu Mac y la app del drone lo va a ofrecer primero.

## Celda 1 — Instalar ultralytics (la librería de YOLO)

In [ ]:
# Instala la librería que entrena el modelo. Tarda ~1 minuto.
%pip install -q ultralytics
import ultralytics
print('Listo. Ultralytics', ultralytics.__version__)

## Celda 2 — Subir y descomprimir el dataset

Cuando aparezca el botón **Elegir archivos**, seleccioná `dataset_bovinos_limpio.zip` (está en la carpeta del proyecto en tu Mac). La subida puede tardar varios minutos según tu internet.

In [ ]:
import zipfile
from pathlib import Path

if not Path('dataset_bovinos_limpio.zip').exists():
    from google.colab import files
    print('Elegí el archivo dataset_bovinos_limpio.zip de tu computadora…')
    files.upload()

with zipfile.ZipFile('dataset_bovinos_limpio.zip') as z:
    z.extractall('.')

# El dataset.yaml trae la ruta de la Mac: la corregimos para Colab.
import yaml
p = Path('dataset_bovinos_limpio/dataset.yaml')
cfg = yaml.safe_load(p.read_text())
cfg['path'] = str(Path('dataset_bovinos_limpio').resolve())
p.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False))

n_train = len(list(Path('dataset_bovinos_limpio/images/train').glob('*.jpg')))
n_val = len(list(Path('dataset_bovinos_limpio/images/val').glob('*.jpg')))
print(f'Dataset listo: {n_train} imágenes de entrenamiento, {n_val} de validación.')

## Celda 3 — Entrenar el modelo

Parte del modelo `yolov8m` pre-entrenado y lo especializa en bovinos vistos desde arriba. Con `degrees=180` le enseñamos que en vista cenital los animales pueden apuntar hacia cualquier lado. Corta solo si deja de mejorar (`patience=15`). **Tarda 30–90 min: dejá la pestaña abierta.**

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8m.pt')  # detect (sin máscaras): más rápido y preciso para contar
model.train(
    data='dataset_bovinos_limpio/dataset.yaml',
    epochs=60,        # vueltas completas al dataset
    imgsz=1280,       # resolución de entrenamiento (animales chicos -> alta)
    batch=-1,         # tamaño de lote automático según la GPU
    patience=15,      # frena si 15 épocas seguidas no mejora
    degrees=180,      # rotación libre: en nadir la orientación es cualquiera
    project='runs_bovinos',
    name='finetune',
    exist_ok=True,
)
print('Entrenamiento terminado. Mejor modelo: runs_bovinos/finetune/weights/best.pt')

## Celda 4 — Validar: métricas y 3 ejemplos

**mAP50** es la nota del modelo (0 a 1): arriba de 0.80 es muy bueno para contar. Abajo se muestran 3 imágenes de validación con lo que el modelo detecta.

In [ ]:
import random
from pathlib import Path
from ultralytics import YOLO
from PIL import Image
from IPython.display import display

model = YOLO('runs_bovinos/finetune/weights/best.pt')

m = model.val(data='dataset_bovinos_limpio/dataset.yaml', imgsz=1280)
print(f'mAP50: {m.box.map50:.3f}   mAP50-95: {m.box.map:.3f}   '
      f'Precisión: {m.box.mp:.3f}   Recall: {m.box.mr:.3f}')

val_imgs = sorted(Path('dataset_bovinos_limpio/images/val').glob('*.jpg'))
random.seed(0)
for p in random.sample(val_imgs, min(3, len(val_imgs))):
    r = model.predict(str(p), conf=0.25, imgsz=1280, verbose=False)[0]
    im = Image.fromarray(r.plot()[:, :, ::-1])  # BGR -> RGB
    im.thumbnail((960, 960))
    print(f'\n{p.name}: {len(r.boxes)} bovinos detectados')
    display(im)

## Celda 5 — Descargar `hms_bovinos.pt`

Renombra el mejor modelo y lo descarga a tu computadora. **Movelo a la raíz de la carpeta del proyecto** (al lado de `drone_app.py`): la app lo va a mostrar primero en el selector de modelos.

Ojo: `hms_bovinos.pt` es un modelo de *detección* (sin máscaras). Es el mejor para **contar**; para estimar **peso** seguí usando `yolov8l-seg.pt`.

In [ ]:
import shutil
shutil.copy('runs_bovinos/finetune/weights/best.pt', 'hms_bovinos.pt')
from google.colab import files
files.download('hms_bovinos.pt')
print('Descargando hms_bovinos.pt… guardalo en la raíz del proyecto en tu Mac.')